## QAOA Job Scheduler — Database Setup

This notebook creates the IBM i database objects required by the QAOA Job Scheduler.
Run this **once** before running `qaoa_scheduler.ipynb` or `qaoa_scheduler_ibm.ipynb`.

**What this notebook creates:**
- Schema `QSCHEDULE` (if not already present)
- Table function `QSCHEDULE.JOBQUEUE()` — returns sample batch jobs with resource constraints
- Table `QSCHEDULE.SCHEDULE_RESULTS` — stores optimised schedules written back from the solver

**Author:** Quantum-IBMi-Docker Project — QAOA Scheduler

### Import Required Libraries

In [1]:
import getpass
from mapepire_python.client.sql_job import SQLJob
from mapepire_python.data_types import DaemonServer
from IPython.display import Markdown, display

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


### Connect to IBM i via Mapepire

In [2]:
host = input("Enter IBM i hostname/IP: ").strip()
username = input("Enter username: ").strip()
password = getpass.getpass("Enter password: ")
port_input = input("Enter port (press Enter for default 8076): ").strip()
port = int(port_input) if port_input else 8076

daemon_server = DaemonServer(
    host=host,
    port=port,
    user=username,
    password=password
)

sql_job = SQLJob()
sql_job.connect(daemon_server)

display(Markdown("**✓ Connected to IBM i database**"))

Enter IBM i hostname/IP:  idevphp.idevcloud.com
Enter username:  Jgorzins
Enter password:  ········
Enter port (press Enter for default 8076):  


**✓ Connected to IBM i database**

### Verify Connection

In [3]:
query = sql_job.query("SELECT CURRENT_USER FROM SYSIBM.SYSDUMMY1")
result = query.run()

if result.get("success") and result.get("data"):
    current_user = list(result["data"][0].values())[0]
    print(f"✓ Connected as: {current_user}")
else:
    print(f"✗ Query failed: {result.get('error', 'Unknown error')}")

✓ Connected as: JGORZINS


### Create Schema QSCHEDULE

In [4]:
try:
    result = sql_job.query("CREATE SCHEMA QSCHEDULE").run()
    if result.get("success"):
        print("✓ Schema QSCHEDULE created")
    else:
        print(f"Note: {result.get('error', 'Schema may already exist')}")
except Exception as e:
    print(f"Note: {e}")

Note: {'error': '[SQL0601] QSCHEDULE in *N type *LIB already exists.', 'sql_state': '42710', 'sql_rc': -601}


### Create JOBQUEUE() Table Function

Returns a representative set of IBM i batch jobs with:
- `JOB_ID` — unique job identifier (1–8)
- `JOB_NAME` — descriptive name (e.g. PAYROLL, INVUPD)
- `DURATION` — estimated run time in minutes
- `PRIORITY` — business priority 1 (highest) to 5 (lowest)
- `MEM_MB` — memory requirement in MB
- `DEPENDS_ON` — job ID this job must wait for (0 = no dependency)

In [5]:
create_fn_sql = """
CREATE OR REPLACE FUNCTION QSCHEDULE.JOBQUEUE()
  RETURNS TABLE (
    JOB_ID     INT,
    JOB_NAME   VARCHAR(20),
    DURATION   INT,
    PRIORITY   INT,
    MEM_MB     INT,
    DEPENDS_ON INT
  )
  LANGUAGE SQL
  SPECIFIC QSCHEDULE.JOBQUEUE
  NOT DETERMINISTIC
  NO EXTERNAL ACTION
  RETURN
    VALUES
      (1, 'PAYROLL',   45, 1, 512, 0),
      (2, 'INVUPD',    20, 2, 256, 0),
      (3, 'GLCLOSE',   60, 1, 768, 1),
      (4, 'ARAGING',   30, 3, 256, 2),
      (5, 'SALESRPT',  15, 4, 128, 0),
      (6, 'BACKUPDB',  90, 5, 128, 3),
      (7, 'TAXCALC',   35, 1, 512, 1),
      (8, 'SHIPCONF',  10, 2, 128, 0)
"""

try:
    result = sql_job.query(create_fn_sql).run()
    if result.get("success"):
        print("✓ Function QSCHEDULE.JOBQUEUE() created successfully")
        print("  8 sample batch jobs defined")
    else:
        print(f"✗ Failed: {result.get('error', 'Unknown error')}")
except Exception as e:
    print(f"✗ Error: {e}")

✓ Function QSCHEDULE.JOBQUEUE() created successfully
  8 sample batch jobs defined


### Create SCHEDULE_RESULTS Table

Stores the optimised job order returned by the QAOA solver, along with metadata
about the run (solver type, iterations, total makespan).

In [6]:
create_results_sql = """
CREATE OR REPLACE TABLE QSCHEDULE.SCHEDULE_RESULTS (
    RUN_ID        INT GENERATED ALWAYS AS IDENTITY,
    RUN_TS        TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    SOLVER_TYPE   VARCHAR(20),
    BACKEND       VARCHAR(50),
    QAOA_LAYERS   INT,
    SHOTS         INT,
    JOB_ORDER     VARCHAR(200),
    MAKESPAN_MIN  INT,
    TOP_BITSTRING VARCHAR(64),
    TOP_COUNT     INT,
    PRIMARY KEY (RUN_ID)
)
"""

try:
    result = sql_job.query(create_results_sql).run()
    if result.get("success"):
        print("✓ Table QSCHEDULE.SCHEDULE_RESULTS created successfully")
    else:
        print(f"Note: {result.get('error', 'Table may already exist')}")
except Exception as e:
    print(f"Note: {e}")

✓ Table QSCHEDULE.SCHEDULE_RESULTS created successfully


### Verify JOBQUEUE() Function

In [7]:
query = sql_job.query("SELECT * FROM TABLE(QSCHEDULE.JOBQUEUE()) AS t ORDER BY JOB_ID")
result = query.run()

if result.get("success") and result.get("data"):
    print(f"✓ JOBQUEUE() returns {len(result['data'])} jobs\n")
    print(f"{'ID':<4} {'NAME':<12} {'DURATION':>8} {'PRIORITY':>8} {'MEM_MB':>8} {'DEPENDS_ON':>10}")
    print("-" * 55)
    for row in result["data"]:
        print(f"{row['JOB_ID']:<4} {row['JOB_NAME']:<12} {row['DURATION']:>8} "
              f"{row['PRIORITY']:>8} {row['MEM_MB']:>8} {row['DEPENDS_ON']:>10}")
else:
    print(f"✗ Verification failed: {result.get('error', 'No data')}")

✓ JOBQUEUE() returns 8 jobs

ID   NAME         DURATION PRIORITY   MEM_MB DEPENDS_ON
-------------------------------------------------------
1    PAYROLL            45        1      512          0
2    INVUPD             20        2      256          0
3    GLCLOSE            60        1      768          1
4    ARAGING            30        3      256          2
5    SALESRPT           15        4      128          0
6    BACKUPDB           90        5      128          3
7    TAXCALC            35        1      512          1
8    SHIPCONF           10        2      128          0


### Verify SCHEDULE_RESULTS Table

In [8]:
query = sql_job.query(
    "SELECT COLUMN_NAME, DATA_TYPE, LENGTH "
    "FROM QSYS2.SYSCOLUMNS "
    "WHERE TABLE_SCHEMA = 'QSCHEDULE' AND TABLE_NAME = 'SCHEDULE_RESULTS' "
    "ORDER BY ORDINAL_POSITION"
)
result = query.run()

if result.get("success") and result.get("data"):
    print("✓ SCHEDULE_RESULTS columns:")
    for col in result["data"]:
        print(f"  {col['COLUMN_NAME']:<20} {col['DATA_TYPE']:<15} {col.get('LENGTH', '')}")
else:
    print(f"Note: {result.get('error', 'Could not verify table')}")

✓ SCHEDULE_RESULTS columns:
  RUN_ID               INTEGER         4
  RUN_TS               TIMESTMP        10
  SOLVER_TYPE          VARCHAR         20
  BACKEND              VARCHAR         50
  QAOA_LAYERS          INTEGER         4
  SHOTS                INTEGER         4
  JOB_ORDER            VARCHAR         200
  MAKESPAN_MIN         INTEGER         4
  TOP_BITSTRING        VARCHAR         64
  TOP_COUNT            INTEGER         4


### Close Connection

In [9]:
sql_job.close()
print("✓ Connection closed")

✓ Connection closed


### Summary

This notebook has:
1. Connected to IBM i via Mapepire
2. Created the `QSCHEDULE` schema
3. Created the `QSCHEDULE.JOBQUEUE()` table function with 8 sample batch jobs
4. Created the `QSCHEDULE.SCHEDULE_RESULTS` table for storing solver output
5. Verified both objects return correct data

You can now run:
- **`qaoa_scheduler.ipynb`** — solve the scheduling problem on the local Aer simulator
- **`qaoa_scheduler_ibm.ipynb`** — solve on real IBM Quantum hardware